# CLaRa — Evaluate Fine-Tuned Model on TriviaQA (Kaggle T4)

> **Paper:** *CLaRa: Bridging Retrieval and Generation with Continuous Latent Reasoning*  
> He et al., Apple / University of Edinburgh — February 2026  
> **Checkpoint:** `tokiggle/clara-ft-triviaqa` (fine-tuned, uploaded as Kaggle dataset)

---

## Overview

Evaluates the TriviaQA-fine-tuned CLaRa checkpoint on the TriviaQA validation set.

| Setting | Value |
|---------|-------|
| Eval samples | 250 |
| Eval mode | oracle |
| Metrics | EM + F1 |
| Est. time | ~1h |

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 0-A  │  Clone repository & install dependencies
# Run ONCE. After this cell completes → Restart kernel.
# ═══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

REPO_URL  = "https://github.com/Duy-Tuyen/introml-clara-implementation.git"
REPO_NAME = "introml-clara-implementation"
REPO_ROOT = f"/kaggle/working/{REPO_NAME}"

print("[1/3] Cloning repository...")
subprocess.run(["rm", "-rf", REPO_ROOT], check=True)
subprocess.run(["git", "clone", "-b", "feature-tuyen2", REPO_URL, REPO_ROOT], check=True)

print("[2/3] Installing dependencies...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r",
     f"{REPO_ROOT}/requirements.txt", "-q"],
    check=True,
)

print("[3/3] Running setup script...")
subprocess.run([sys.executable, f"{REPO_ROOT}/setup_env.py"], check=True)

print("\n Setup complete. Please RESTART the kernel before continuing.")

In [ ]:
import shutil, os

cache_path = "/root/.cache/huggingface/modules/transformers_modules"
if os.path.exists(cache_path):
    print("Nuking stale code cache...")
    shutil.rmtree(cache_path)
    print("Cache destroyed.")
else:
    print("Cache was already empty.")

In [ ]:
import os, sys

REPO_ROOT = "/kaggle/working/introml-clara-implementation"
assert os.path.isdir(REPO_ROOT), f"Repository not found at {REPO_ROOT}. Run Cell 0-A first."

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print(f"Working directory : {os.getcwd()}")
print(f"Python path entry : {sys.path[0]}")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU: Settings > Accelerator > T4.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU   : {gpu_name}")
print(f"VRAM  : {vram_gb:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1  │  Evaluate fine-tuned model on TriviaQA
# ═══════════════════════════════════════════════════════════════════════════════

import os, subprocess

FT_TRIVIAQA_CKPT = "/kaggle/input/datasets/tokiggle/clara-ft-triviaqa"

print("╔" + "═" * 60 + "╗")
print("║  Evaluate Fine-Tuned Model on TriviaQA                     ║")
print("╠" + "═" * 60 + "╣")
print("║  Checkpoint : tokiggle/clara-ft-triviaqa (fine-tuned)      ║")
print("║  Eval mode  : oracle  |  Metrics: EM + F1                   ║")
print("║  Val samples: 250                                           ║")
print("╚" + "═" * 60 + "╝")

eval_env = os.environ.copy()
eval_env.update({
    "CLARA_CKPT_PATH"     : FT_TRIVIAQA_CKPT,
    "CLARA_DATASET"       : "triviaqa",
    "CLARA_EVAL_MODE"     : "oracle",
    "CLARA_EVAL_BS"       : "1",
    "CLARA_N_VAL"         : "250",
    "CLARA_MODEL_VERSION" : "ModelB_FineTuned_TriviaQA",
    "CLARA_MAX_NEW_TOKENS": "32",
})

subprocess.run(
    ["python", "-m", "scripts.evaluate_apple"],
    env=eval_env, check=True,
)

print("\n✅ TriviaQA evaluation complete. Results saved to results/eval_scores.csv")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2  │  Display results
# ═══════════════════════════════════════════════════════════════════════════════

import os
import pandas as pd

CSV_PATH = "results/eval_scores.csv"

if not os.path.exists(CSV_PATH):
    print(f"No results found at {CSV_PATH}.")
else:
    df = pd.read_csv(CSV_PATH)

    print("═" * 80)
    print("  CLaRa EVALUATION RESULTS — Fine-Tuned TriviaQA")
    print("═" * 80)
    print()

    display_df = df[[
        'Model_Version', 'Dataset', 'Eval_Mode',
        'Exact_Match(%)', 'F1_Score(%)', 'Timestamp'
    ]].copy()

    pd.set_option('display.max_colwidth', 45)
    pd.set_option('display.width', 120)
    print(display_df.to_string(index=False))